In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
df = pd.read_csv('data/ThoraricSurgery.csv')
df

,DGN,PRE4,PRE5,PRE6,PRE7,PRE8,PRE9,PRE10,PRE11,PRE14,PRE17,PRE19,PRE25,PRE30,PRE32,AGE,Risk1Yr
0,DGN2,2.88,2.16,PRZ1,F,F,F,T,T,OC14,F,F,F,T,F,60,F
1,DGN3,3.40,1.88,PRZ0,F,F,F,F,F,OC12,F,F,F,T,F,51,F
2,DGN3,2.76,2.08,PRZ1,F,F,F,T,F,OC11,F,F,F,T,F,59,F
3,DGN3,3.68,3.04,PRZ0,F,F,F,F,F,OC11,F,F,F,F,F,54,F
4,DGN3,2.44,0.96,PRZ2,F,T,F,T,T,OC11,F,F,F,T,F,73,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,DGN2,3.88,2.12,PRZ1,F,F,F,T,F,OC13,F,F,F,T,F,63,F
466,DGN3,3.76,3.12,PRZ0,F,F,F,F,F,OC11,F,F,F,T,F,61,F
467,DGN3,3.04,2.08,PRZ1,F,F,F,T,F,OC13,F,F,F,F,F,52,F
468,DGN3,1.96,1.68,PRZ1,F,F,F,T,T,OC12,F,F,F,T,F,79,F


### 데이터 변환

In [3]:
# DGN
encoder = OneHotEncoder()
encoder.fit(df[['DGN']])
df_encoded = encoder.transform(df[['DGN']]).toarray()

columns = encoder.get_feature_names_out(['DGN'])

encoded_df = pd.DataFrame(df_encoded, columns=columns)
df = pd.concat([df, encoded_df], axis=1)
df.drop('DGN', axis=1, inplace=True)
df

,PRE4,PRE5,PRE6,PRE7,PRE8,PRE9,PRE10,PRE11,PRE14,PRE17,...,PRE32,AGE,Risk1Yr,DGN_DGN1,DGN_DGN2,DGN_DGN3,DGN_DGN4,DGN_DGN5,DGN_DGN6,DGN_DGN8
0,2.88,2.16,PRZ1,F,F,F,T,T,OC14,F,...,F,60,F,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,3.40,1.88,PRZ0,F,F,F,F,F,OC12,F,...,F,51,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2.76,2.08,PRZ1,F,F,F,T,F,OC11,F,...,F,59,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3.68,3.04,PRZ0,F,F,F,F,F,OC11,F,...,F,54,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,2.44,0.96,PRZ2,F,T,F,T,T,OC11,F,...,F,73,T,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,3.88,2.12,PRZ1,F,F,F,T,F,OC13,F,...,F,63,F,0.0,1.0,0.0,0.0,0.0,0.0,0.0
466,3.76,3.12,PRZ0,F,F,F,F,F,OC11,F,...,F,61,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0
467,3.04,2.08,PRZ1,F,F,F,T,F,OC13,F,...,F,52,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0
468,1.96,1.68,PRZ1,F,F,F,T,T,OC12,F,...,F,79,F,0.0,0.0,1.0,0.0,0.0,0.0,0.0


### PRE6

In [4]:
encoder = OneHotEncoder()
encoder.fit(df[['PRE6']])
df_encoded = encoder.transform(df[['PRE6']]).toarray()

columns = encoder.get_feature_names_out(['PRE6'])

encoded_df = pd.DataFrame(df_encoded, columns=columns)
df = pd.concat([df, encoded_df], axis=1)
df.drop('PRE6', axis=1, inplace=True)
df

,PRE4,PRE5,PRE7,PRE8,PRE9,PRE10,PRE11,PRE14,PRE17,PRE19,...,DGN_DGN1,DGN_DGN2,DGN_DGN3,DGN_DGN4,DGN_DGN5,DGN_DGN6,DGN_DGN8,PRE6_PRZ0,PRE6_PRZ1,PRE6_PRZ2
0,2.88,2.16,F,F,F,T,T,OC14,F,F,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,3.40,1.88,F,F,F,F,F,OC12,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2.76,2.08,F,F,F,T,F,OC11,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3.68,3.04,F,F,F,F,F,OC11,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,2.44,0.96,F,T,F,T,T,OC11,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,3.88,2.12,F,F,F,T,F,OC13,F,F,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
466,3.76,3.12,F,F,F,F,F,OC11,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
467,3.04,2.08,F,F,F,T,F,OC13,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
468,1.96,1.68,F,F,F,T,T,OC12,F,F,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### PRE14

In [5]:
encoder = OneHotEncoder()
encoder.fit(df[['PRE14']])
df_encoded = encoder.transform(df[['PRE14']]).toarray()

columns = encoder.get_feature_names_out(['PRE14'])

encoded_df = pd.DataFrame(df_encoded, columns=columns)
df = pd.concat([df,encoded_df], axis=1)
df.drop('PRE14', axis=1, inplace=True)
df

,PRE4,PRE5,PRE7,PRE8,PRE9,PRE10,PRE11,PRE17,PRE19,PRE25,...,DGN_DGN5,DGN_DGN6,DGN_DGN8,PRE6_PRZ0,PRE6_PRZ1,PRE6_PRZ2,PRE14_OC11,PRE14_OC12,PRE14_OC13,PRE14_OC14
0,2.88,2.16,F,F,F,T,T,F,F,F,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,3.40,1.88,F,F,F,F,F,F,F,F,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2.76,2.08,F,F,F,T,F,F,F,F,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,3.68,3.04,F,F,F,F,F,F,F,F,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2.44,0.96,F,T,F,T,T,F,F,F,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,3.88,2.12,F,F,F,T,F,F,F,F,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
466,3.76,3.12,F,F,F,F,F,F,F,F,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
467,3.04,2.08,F,F,F,T,F,F,F,F,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
468,1.96,1.68,F,F,F,T,T,F,F,F,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### True/False => 1/0 변환

In [6]:
df['PRE7'] = df['PRE7'].map({'F':0,'T':1})
df['PRE8'] = df['PRE8'].map({'F':0,'T':1})
df['PRE9'] = df['PRE9'].map({'F':0,'T':1})
df['PRE10'] = df['PRE10'].map({'F':0,'T':1})
df['PRE11'] = df['PRE11'].map({'F':0,'T':1})
df['PRE17'] = df['PRE17'].map({'F':0,'T':1})
df['PRE19'] = df['PRE19'].map({'F':0,'T':1})
df['PRE25'] = df['PRE25'].map({'F':0,'T':1})
df['PRE30'] = df['PRE30'].map({'F':0,'T':1})
df['PRE32'] = df['PRE32'].map({'F':0,'T':1})
df['Risk1Yr'] = df['Risk1Yr'].map({'F':0,'T':1})
df

,PRE4,PRE5,PRE7,PRE8,PRE9,PRE10,PRE11,PRE17,PRE19,PRE25,...,DGN_DGN5,DGN_DGN6,DGN_DGN8,PRE6_PRZ0,PRE6_PRZ1,PRE6_PRZ2,PRE14_OC11,PRE14_OC12,PRE14_OC13,PRE14_OC14
0,2.88,2.16,0,0,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,3.40,1.88,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2.76,2.08,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,3.68,3.04,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2.44,0.96,0,1,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,3.88,2.12,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
466,3.76,3.12,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
467,3.04,2.08,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
468,1.96,1.68,0,0,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [7]:
df.to_csv('data/ThoraricSurgery2.csv', index=False)